# NB19 화폐의 기능론 — 19장

『법률가를 위한 블록체인 — 기술과 경제학으로 읽기』의 장별 노트북입니다. 본문의 코드와 같습니다.

- 모든 코드는 원리를 눈으로 보기 위한 **장난감 구현**입니다. 실제 자산·실제 개인키·실제 거래에 쓰지 마십시오(실사용 금지).
- 외부 API를 부르지 않습니다. 네트워크 없이 실행됩니다. 동봉 스냅샷 데이터가 필요한 셀은 그 데이터를 셀 안에 적어 두었습니다.
- 표준 라이브러리만 씁니다(Python 3.13에서 실행·확인). 설치 셀은 없습니다.
- 출력은 엮은이가 실행한 결과를 그대로 담아 두었습니다. 다시 실행하면 같은 값이 나와야 합니다(난수는 시드를 고정했습니다).

In [ ]:
# NB19-1: 개럿–월리스 질문 (iv) — 매 기 확률 1−π 로 오는 붕괴. 「언젠가」의 확률과 「바로 다음 기」의 확률은 다르다 (공부용)
pi = 0.99                                            # 한 기를 더 살아남을 확률(논문의 예)
print(f"π = {pi}: 다음 한 기에 붕괴할 조건부 확률 = {1 - pi:.2f}")
print(f"{'n(기)':>6} {'n기 안에 붕괴할 확률 1−π^n':>26} {'n기까지 살아남을 확률 π^n':>26}")
for n in [1, 10, 30, 68, 100, 300, 1000]:
    print(f"{n:>6} {1 - pi**n:>26.3f} {pi**n:>26.3f}")
print("\n68기 안에 붕괴할 확률은 거의 절반이지만, 67기에서 본 68기의 붕괴 확률은 여전히 0.01이다(논문 1891–1892쪽의 산수).")
print("어느 기에 붕괴하는가의 분포는 기하분포이고, 그 기대값은", round(1 / (1 - pi)), "기 — 평균은 있지만 날짜는 없다.")

π = 0.99: 다음 한 기에 붕괴할 조건부 확률 = 0.01
  n(기)         n기 안에 붕괴할 확률 1−π^n           n기까지 살아남을 확률 π^n
     1                      0.010                      0.990
    10                      0.096                      0.904
    30                      0.260                      0.740
    68                      0.495                      0.505
   100                      0.634                      0.366
   300                      0.951                      0.049
  1000                      1.000                      0.000

68기 안에 붕괴할 확률은 거의 절반이지만, 67기에서 본 68기의 붕괴 확률은 여전히 0.01이다(논문 1891–1892쪽의 산수).
어느 기에 붕괴하는가의 분포는 기하분포이고, 그 기대값은 100 기 — 평균은 있지만 날짜는 없다.


In [ ]:
# NB19-2: 실링–울리히 식 (8) — 초기값 Q0 와 평균 0 교란 ε 를 「고르면」 균형 경로가 하나 만들어진다. 고를 수 있는 만큼 균형이 있다 (공부용)
# 주의: Q 는 모형 변수(단위 없음)다. 어떤 자산의 가격도 아니고, 아래 경로는 전망이 아니다.
import random
random.seed(19)

def construct_path(Q0, steps, eps_draw, cov_term=0.0):
    """식 (8): Q_{t+1} = Q_t + ε_{t+1} − cov_t(ε_{t+1}, m_{t+1}) / E_t[m_{t+1}].
    cov_term=0 이면 마팅게일(명제 1 식 (2)). 상관이 있으면 cov_term 만큼 기대 변화가 생긴다(따름정리 1)."""
    Q = [Q0]
    for _ in range(steps):
        Q.append(Q[-1] + eps_draw() - cov_term)
    return Q

Q0, steps = 100.0, 12
draws = {
    "ε≡0 (카레켄–월리스의 일정 가격)": lambda: 0.0,
    "ε ~ 균등(−5, 5)":               lambda: random.uniform(-5, 5),
    "ε ∈ {−20, +20} 각 1/2":          lambda: random.choice([-20.0, 20.0]),
}
print(f"Q0 = {Q0}, {steps}기. 같은 Q0 에서 출발한 세 균형 경로(마팅게일, cov 항 0):")
for name, d in draws.items():
    path = construct_path(Q0, steps, d)
    print(f"  {name:<32} → " + " ".join(f"{q:6.1f}" for q in path))

# 마팅게일 성질의 수치 확인: 많은 경로의 평균은 Q0 근처에 머문다(기대값으로 오르지도 내리지도 않는다)
N = 20000
end_values = [construct_path(Q0, steps, draws["ε ~ 균등(−5, 5)"])[-1] for _ in range(N)]
print(f"\n균등 교란 경로 {N:,}개의 {steps}기 말 평균: {sum(end_values)/N:.2f} (Q0 = {Q0}) — 평균은 그대로, 흩어짐만 커진다")
print(f"  {steps}기 말 최소 {min(end_values):.1f}, 최대 {max(end_values):.1f}")

# 따름정리 1: 가격 커널과의 상관이 양(+)이면 슈퍼마팅게일(기대값으로 하락), 음(−)이면 서브마팅게일(상승). 부호는 모형이 정하지 않는다.
for cov_term, label in [(1.0, "cov > 0 → 슈퍼마팅게일"), (-1.0, "cov < 0 → 서브마팅게일")]:
    mean_end = sum(construct_path(Q0, steps, draws["ε ~ 균등(−5, 5)"], cov_term)[-1] for _ in range(N)) / N
    print(f"  {label}: {steps}기 말 평균 {mean_end:.2f}")

Q0 = 100.0, 12기. 같은 Q0 에서 출발한 세 균형 경로(마팅게일, cov 항 0):
  ε≡0 (카레켄–월리스의 일정 가격)             →  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0  100.0
  ε ~ 균등(−5, 5)                    →  100.0  101.8  104.6  104.8  104.9  103.9  108.8  106.7  103.2  100.8   98.4   96.7   94.4
  ε ∈ {−20, +20} 각 1/2             →  100.0   80.0  100.0  120.0  100.0   80.0   60.0   40.0   20.0   40.0   60.0   40.0   20.0

균등 교란 경로 20,000개의 12기 말 평균: 99.94 (Q0 = 100.0) — 평균은 그대로, 흩어짐만 커진다
  12기 말 최소 64.7, 최대 138.4
  cov > 0 → 슈퍼마팅게일: 12기 말 평균 87.89
  cov < 0 → 서브마팅게일: 12기 말 평균 112.12
